In [20]:
from itertools import product
from pydantic import BaseModel
from typing import List
import numpy as np

In [33]:
# Payment Channel Features
# out of all the frauds, this is the percentage of frauds that are of each type
payment_channel_fraud = {"cnp": 0.75, "digital_wallet": 0.15, "pos": 0.1}

# Customer Features
# if the customer enables 2FA, the probability of fraud is 0.5% otherwise it is 5%
two_fa_fraud = {1: 0.005, 0: 0.05}

# groups that are more susceptible to frauds are millenials and seniors
age_group_fraud = {
    "18-25": 0.05,
    "26-35": 0.05,
    "36-45": 0.01,
    "46-55": 0.01,
    "56-65": 0.05,
    "66+": 0.05,
}

# groups with increasingly higher spending are more susceptible to frauds:
spending_group_fraud = {
    "low": 0.005,
    "low-middle": 0.0075,
    "midle": 0.01,
    "middle-high": 0.0125,
    "high": 0.015,
}


class FeatureConfig(BaseModel):
    name: List[str]
    customer_pct: List[float]
    fraud_proba: List[float]


age_config = {
    "name": ["18-25", "26-35", "36-45", "46-55", "56-65", "66+"],
    "customer_pct": [0.1, 0.15, 0.2, 0.25, 0.15, 0.15],
    "fraud_proba": [0.05, 0.05, 0.01, 0.01, 0.05, 0.05],
}

spending_config = {
    "name": ["low", "low-middle", "midle", "middle-high", "high"],
    "customer_pct": [0.1, 0.25, 0.35, 0.25, 0.05],
    "fraud_proba": [0.005, 0.0075, 0.01, 0.0125, 0.015],
    # "txn_mean_low": [5, 20, 40, 60, 80],
    # "txn_mean_high": [20, 40, 60, 80, 100],
    # "txn_cv_low": [0.3, 0.4, 0.5, 0.6, 0.7],
    # "txn_cv_high": [0.4, 0.5, 0.6, 0.7, 0.8],
    # "txn_lambda": [0.25, 0.5, 1, 1.5, 2],
}

two_fa_config = {
    "name": ["no", "yes"],
    "customer_pct": [0.5, 0.5],
    "fraud_proba": [0.05, 0.005],
}


class CustomerProfileGenerator:
    def __init__(
        self,
        age_config: dict,
        spending_config: dict,
        two_fa_config: dict,
    ):
        self.age_config = age_config
        self.spending_config = spending_config
        self.two_fa_config = two_fa_config

    def generate_profile_with_fraud_exposure(self):
        age_group = np.random.choice(
            self.age_config["name"], p=self.age_config["customer_pct"]
        ).item()

        spending_group = np.random.choice(
            self.spending_config["name"], p=self.spending_config["customer_pct"]
        ).item()

        two_fa = np.random.choice(
            self.two_fa_config["name"], p=self.two_fa_config["customer_pct"]
        ).item()

        age_group_fraud_proba = self.age_config["fraud_proba"][
            self.age_config["name"].index(age_group)
        ]

        spending_group_fraud_proba = self.spending_config["fraud_proba"][
            self.spending_config["name"].index(spending_group)
        ]

        two_fa_fraud_proba = self.two_fa_config["fraud_proba"][
            self.two_fa_config["name"].index(two_fa)
        ]

        profile = {
            "age_group": [age_group, age_group_fraud_proba],
            "spending_group": [spending_group, spending_group_fraud_proba],
            "two_fa": [two_fa, two_fa_fraud_proba],
        }

        return profile


customer_profile_generator = CustomerProfileGenerator(
    age_config, spending_config, two_fa_config
)
customer_profile_generator.generate_profile_with_fraud_exposure()

{'age_group': ['26-35', 0.05],
 'spending_group': ['low-middle', 0.0075],
 'two_fa': ['yes', 0.005]}

In [34]:
# generate 1000 custemer profiles and check the distribution 
profiles = [customer_profile_generator.generate_profile_with_fraud_exposure() for i in range(1000)]
age_groups = [profile["age_group"][0] for profile in profiles]
spending_groups = [profile["spending_group"][0] for profile in profiles]
two_fa = [profile["two_fa"][0] for profile in profiles]

# calculate the distribution of these features
age_group_dist = {age: age_groups.count(age) / 1000 for age in set(age_groups)}
spending_group_dist = {
    spending: spending_groups.count(spending) / 1000 for spending in set(spending_groups)
}
two_fa_dist = {two_fa: two_fa.count(two_fa) / 1000 for two_fa in set(two_fa)}

print(age_group_dist)
print(spending_group_dist)
print(two_fa_dist)

{'56-65': 0.152, '26-35': 0.133, '46-55': 0.26, '36-45': 0.215, '66+': 0.141, '18-25': 0.099}
{'middle-high': 0.231, 'low-middle': 0.239, 'high': 0.056, 'midle': 0.368, 'low': 0.106}
{'yes': 0.001, 'no': 0.001}


In [ ]:
# as these features are usually not independet from each other, e.g., it is reasonable that a senior is more likely to have a high spending, while less tech-savy to enable 2FA, we should not use a multiplication of the probabilities to calculate the final probability of fraud. Instead, we can use a weighted average model:
def weighted_average(two_factor_auth, age_group, spending_group):
    weights = {
        "two_factor_auth": 0.5,
        "spending_group": 0.3,
        "age_group": 0.2,
    }

    probability = (
        weights["two_factor_auth"] * two_factor_auth_fraud[two_factor_auth]
        + weights["age_group"] * age_group_fraud[age_group]
        + weights["spending_group"] * spending_group_fraud[spending_group]
    )
    return probability


sample_probability = weighted_average(1, "66+", "high")
sample_probability

# calculate the probability of all the combinations of the features:
fraud_probabilities = {}
for two_factor_auth in two_factor_auth_fraud:
    for age_group in age_group_fraud:
        for spending_group in spending_group_fraud:
            probability = round(
                weighted_average(two_factor_auth, age_group, spending_group), 6
            )
            fraud_probabilities[(two_factor_auth, age_group, spending_group)] = (
                probability
            )
fraud_probabilities

# for simplicity, we can ignore the relationship between the customer feature such as age and spending groups, or age and 2FA, and split the total number of customers equally among the combinations of the features:
total_customers = 100000

# create a list of all the combinations of the features
# this is equivalent to nested for loops
combinations = list(product(two_factor_auth_fraud, age_group_fraud, spending_group_fraud))

customers_per_combination = total_customers / len(combinations)
customers_per_combination

1666.6666666666667